# 05 — Feature engineering

**Job:** fit all imputing, scaling, and encoding on training only, then apply the fitted
objects unchanged to validation and test. Actual delivery/carrier timestamps and IDs are
excluded to prevent target leakage.

In [2]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / "config.py").exists(): ROOT = ROOT.parent
SOURCE = ROOT / "artifacts" / "03_split"
OUT = ROOT / "artifacts" / "05_features"
OUT.mkdir(parents=True, exist_ok=True)
for name in ("train", "validation", "test"):
    assert (SOURCE / f"{name}.csv").exists(), "Run notebook 03 first"

In [4]:
numeric_features = [
    "item_count", "unique_product_count", "unique_seller_count", "total_item_value",
    "mean_item_price", "total_freight_value", "mean_product_weight_g",
    "mean_product_volume_cm3", "mean_product_photos_qty", "payment_count",
    "payment_type_count", "max_payment_installments", "total_payment_value",
    "seller_customer_distance_km", "estimated_lead_days", "purchase_month",
    "purchase_weekday", "purchase_hour",
]
categorical_features = [
    "customer_state", "primary_seller_state", "primary_product_category",
    "primary_payment_type",
]

def build_features(frame):
    purchase = pd.to_datetime(frame["order_purchase_timestamp"], errors="coerce")
    estimate = pd.to_datetime(frame["order_estimated_delivery_date"], errors="coerce")
    features = frame.copy()
    features["estimated_lead_days"] = (estimate - purchase).dt.total_seconds() / 86400
    features["purchase_month"] = purchase.dt.month
    features["purchase_weekday"] = purchase.dt.weekday
    features["purchase_hour"] = purchase.dt.hour
    return features[numeric_features + categorical_features]

raw = {name: pd.read_csv(SOURCE / f"{name}.csv", low_memory=False)
       for name in ("train", "validation", "test")}
X_raw = {name: build_features(frame) for name, frame in raw.items()}
y = {name: frame["is_late"].to_numpy(dtype=np.int8) for name, frame in raw.items()}

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=10, sparse_output=True)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], sparse_threshold=1.0)

transformed = {"train": preprocessor.fit_transform(X_raw["train"])}
transformed["validation"] = preprocessor.transform(X_raw["validation"])
transformed["test"] = preprocessor.transform(X_raw["test"])

In [5]:
feature_names = preprocessor.get_feature_names_out().tolist()
for name in ("train", "validation", "test"):
    matrix = sparse.csr_matrix(transformed[name])
    assert matrix.shape[0] == len(y[name])
    assert matrix.shape[1] == len(feature_names)
    sparse.save_npz(OUT / f"X_{name}.npz", matrix)
    np.save(OUT / f"y_{name}.npy", y[name])
    raw[name][["order_id"]].to_csv(OUT / f"{name}_order_ids.csv", index=False)

joblib.dump(preprocessor, OUT / "preprocessor.joblib")
(OUT / "feature_names.json").write_text(json.dumps(feature_names, indent=2), encoding="utf-8")
metadata = {
    "fit_split": "train", "numeric_inputs": numeric_features,
    "categorical_inputs": categorical_features, "output_feature_count": len(feature_names),
    "leakage_exclusions": [
        "order_delivered_customer_date", "order_delivered_carrier_date", "order_approved_at",
        "order_id", "customer_id", "customer_unique_id", "is_late",
    ],
}
(OUT / "feature_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print({name: transformed[name].shape for name in transformed})
print(f"Saved fitted preprocessor and {len(feature_names)} output feature names")

{'train': (67529, 146), 'validation': (14470, 146), 'test': (14471, 146)}
Saved fitted preprocessor and 146 output feature names
